In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import rc
import astropy.units as u
import astropy.coordinates as coord
import astropy as ap

df1 = pd.read_csv("globular_clusters.csv")
df1 = df1[0:-3]
harris_p1 = pd.read_csv('HarrisPartI.csv')
harris_p2 = pd.read_csv('HarrisPartII.csv')
harris_p3 = pd.read_csv('HarrisPartIII.csv')


In [2]:

df1["Age0"]=df1["Age"].str.split(" ±").str[0].astype(float)
df1["Age_error"]=df1["Age"].str.split(" ±").str[1].astype(float)
df1["Name"] = "NGC " + df1["NGC"].astype(float).astype("Int64").astype(str)
harris_p2["ID"] = harris_p2["ID"].astype(str).str.strip().str.upper()
df1["Name"] = df1["Name"].astype(str).str.strip().str.upper()
rc('text', usetex=False)
R_gc = pd.to_numeric(harris_p1['R_gc'],errors='coerce')
X = pd.to_numeric(harris_p1['X'],errors='coerce')
Y = pd.to_numeric(harris_p1['Y'],errors='coerce')
Z = pd.to_numeric(harris_p1['Z'],errors='coerce')
Ra = pd.to_numeric(harris_p1['RA'],errors='coerce')
Dec = pd.to_numeric(harris_p1['DEC'],errors='coerce')
R_Sun = pd.to_numeric(harris_p1['R_Sun'],errors='coerce')
sig_v = pd.to_numeric(harris_p3['sig_v'],errors='coerce')
v_r = pd.to_numeric(harris_p3['v_r'],errors='coerce')
mtlc = pd.to_numeric(harris_p2['[Fe/H]'],errors='coerce')
rho_0 = pd.to_numeric(harris_p3['rho_0'],errors='coerce')
mu_V = pd.to_numeric(harris_p3['mu_V'],errors='coerce')
c = pd.to_numeric(harris_p3['c'],errors='coerce')
BmV = pd.to_numeric(harris_p2['B-V'],errors='coerce')
ellip = pd.to_numeric(harris_p2['ellip'],errors='coerce')
r_c = pd.to_numeric(harris_p3['r_c'],errors='coerce')
r_h = pd.to_numeric(harris_p3['r_h'],errors='coerce')
M_V = pd.to_numeric(harris_p2['M_V,t'],errors='coerce')
V_t = pd.to_numeric(harris_p2['V_t'],errors='coerce')
wt = pd.to_numeric(harris_p2['wt'],errors='coerce')
L = pd.to_numeric(harris_p1['L'],errors='coerce')
B = pd.to_numeric(harris_p1['B'],errors='coerce')
v_LSR= pd.to_numeric(harris_p3['v_LSR'],errors='coerce')
mask = mtlc > -0.8
log_tc= pd.to_numeric(harris_p3['lg_tc'],errors='coerce')
log_th= pd.to_numeric(harris_p3['lg_th'],errors='coerce')
## Copied of Tobias
VmR = pd.to_numeric(harris_p2['V-R'],errors='coerce')
UmB = pd.to_numeric(harris_p2['U-B'],errors='coerce')
VmI = pd.to_numeric(harris_p2['V-I'],errors='coerce')
V_hb = pd.to_numeric(harris_p2['V_HB'],errors='coerce')
## Correct colour values for foreground reddening
E_BmV = pd.to_numeric(harris_p2['E(B-V)'],errors='coerce')
BmV0=BmV - E_BmV
VmR0=VmR - 0.72*E_BmV
VmI0=VmI - 1.24*E_BmV
UmB0=UmB - 0.72*E_BmV
#Coefficients calculated from Cardelli et al. 1989, ApJ, 345, 245
Age = pd.to_numeric(df1['Age0'],errors='coerce')
Age_e = pd.to_numeric(df1['Age_error'],errors='coerce') 
#Convert RA and DEC to degrees

Ra_rad = coord.Angle(Ra, unit=u.hourangle).rad
Dec_rad = coord.Angle(Dec, unit=u.degree).rad


In [3]:
## Merge age and harris dataframes
merged_df = harris_p2.merge(
    df1[["Name", "Age0"]],
    left_on="ID",
    right_on="Name",
    how="left",
    validate="many_to_one"
)
merged_df = merged_df.merge(
    harris_p1[["ID", "R_gc"]],
    on="ID",
    how="left",
    validate="many_to_one"
)
merged_df = merged_df.merge(
    harris_p1[["ID", "X", "Y", "Z", "RA", "DEC", "R_Sun" ]],
    on="ID",
    how="left",
    validate="many_to_one"
)
merged_df = merged_df.merge(
    harris_p2[["ID", "M_V,t"]],
    on="ID",
    how="left",
    validate="many_to_one"
)
merged_df['RA'] = coord.Angle(merged_df['RA'], unit=u.hourangle).rad
merged_df['DEC'] = coord.Angle(merged_df['DEC'], unit=u.degree).rad


Saggitarius Stream Stuff. Lots of code taken other people and documents.

In [4]:
#Plotting the Saggitarius Stream. 
from astropy.coordinates import frame_transform_graph, rotation_matrix
class Sagittarius(coord.BaseCoordinateFrame):
    """A Heliocentric spherical coordinate system defined by the orbit
    of the Sagittarius dwarf galaxy, as described in
        https://ui.adsabs.harvard.edu/abs/2003ApJ...599.1082M
    and further explained in
        https://www.stsci.edu/~dlaw/Sgr/.

    Parameters
    ----------
    representation : `~astropy.coordinates.BaseRepresentation` or None
        A representation object or None to have no data (or use the other keywords)
    Lambda : `~astropy.coordinates.Angle`, optional, must be keyword
        The longitude-like angle corresponding to Sagittarius' orbit.
    Beta : `~astropy.coordinates.Angle`, optional, must be keyword
        The latitude-like angle corresponding to Sagittarius' orbit.
    distance : `~astropy.units.Quantity`, optional, must be keyword
        The Distance for this object along the line-of-sight.
    pm_Lambda_cosBeta : `~astropy.units.Quantity`, optional, must be keyword
        The proper motion along the stream in ``Lambda`` (including the
        ``cos(Beta)`` factor) for this object (``pm_Beta`` must also be given).
    pm_Beta : `~astropy.units.Quantity`, optional, must be keyword
        The proper motion in Declination for this object (``pm_ra_cosdec`` must
        also be given).
    radial_velocity : `~astropy.units.Quantity`, optional, keyword-only
        The radial velocity of this object.
    """
    default_representation = coord.SphericalRepresentation
    default_differential = coord.SphericalCosLatDifferential
    frame_specific_representation_info = {
        coord.SphericalRepresentation: [
            coord.RepresentationMapping("lon", "Lambda"),
            coord.RepresentationMapping("lat", "Beta"),
            coord.RepresentationMapping("distance", "distance"),
        ]
    }
SGR_PHI = (180 + 3.75) * u.degree  # Euler angles (from Law & Majewski 2010)
SGR_THETA = (90 - 13.46) * u.degree
SGR_PSI = (180 + 14.111534) * u.degree
SGR_MATRIX = (
    np.diag([1.0, 1.0, -1.0])
    @ rotation_matrix(SGR_PSI, "z")
    @ rotation_matrix(SGR_THETA, "x")
    @ rotation_matrix(SGR_PHI, "z")
)
@frame_transform_graph.transform(
    coord.StaticMatrixTransform, coord.Galactic, Sagittarius
)
def galactic_to_sgr():
    """Compute the Galactic spherical to heliocentric Sgr transformation matrix."""
    return SGR_MATRIX
@frame_transform_graph.transform(
    coord.StaticMatrixTransform, Sagittarius, coord.Galactic
)
def sgr_to_galactic():
    """Compute the heliocentric Sgr to spherical Galactic transformation matrix."""
    return SGR_MATRIX.swapaxes(-2, -1)

    ##Code sourced from Astropy document "Create a new coordinate class (for the Sagittarius stream)"


In [8]:

from astropy.coordinates import SkyCoord
from gala.coordinates import SgrCoordinates

sgr = SgrCoordinates()

# Use the dataframe that contains your converted coordinates
gc_data = merged_df.dropna(subset=["RA", "DEC"]).copy()

# Create ICRS coordinates (RA and DEC are already in radians)
gc_icrs = SkyCoord(
    ra=gc_data["RA"].to_numpy(dtype=float) * u.rad,
    dec=gc_data["DEC"].to_numpy(dtype=float) * u.rad,
    frame="icrs"
)

# Transform GCs into Sagittarius coordinates
gc_sgr = gc_icrs.transform_to(sgr.frame)

fig, axes = plt.subplots(
    2, 1, figsize=(8, 10),
    subplot_kw={"projection": "aitoff"}
)

# Top: Sagittarius coordinates
axes[0].set_title("Sagittarius")
axes[0].plot(
    sgr.Lambda.wrap_at(180 * u.deg).radian,
    sgr.Beta.radian,
    linestyle="none", marker=".", markersize=2,
    label="Sagittarius stream"
)
axes[0].scatter(
    gc_sgr.Lambda.wrap_at(180 * u.deg).radian,
    gc_sgr.Beta.radian,
    marker="*", s=45, color="red",
    label="Globular clusters", zorder=3
)
axes[0].grid(True)
axes[0].legend()

# Bottom: ICRS coordinates
axes[1].set_title("ICRS")
axes[1].plot(
    icrs.ra.wrap_at(180 * u.deg).radian,
    icrs.dec.radian,
    linestyle="none", marker=".", markersize=2,
    label="Sagittarius stream"
)
axes[1].scatter(
    gc_icrs.ra.wrap_at(180 * u.deg).radian,
    gc_icrs.dec.radian,
    marker="*", s=45, color=merged_df["[Fe/H]"],
    label="Globular clusters", zorder=3
)
axes[1].grid(True)
axes[1].legend()

# Galactic Centre in ICRS coordinates
galactic_centre = SkyCoord(
    ra=266.4051 * u.deg,
    dec=-28.9362 * u.deg,
    frame="icrs"
)

# Transform into Sagittarius coordinates
gc_centre_sgr = galactic_centre.transform_to(sgr.frame)

axes[0].scatter(
    gc_centre_sgr.Lambda.wrap_at(180 * u.deg).radian,
    gc_centre_sgr.Beta.radian,
    marker="X",
    s=150,
    color="gold",
    edgecolors="black",
    label="Galactic Centre",
    zorder=5
)
axes[1].scatter(
    galactic_centre.ra.wrap_at(180 * u.deg).radian,
    galactic_centre.dec.radian,
    marker="X",
    s=150,
    color="gold",
    edgecolors="black",
    label="Galactic Centre",
    zorder=5
)

plt.tight_layout()
plt.show()








ModuleNotFoundError: No module named 'gala'

In [7]:

# Set missing distances to NaN
projected_dist[~np.isfinite(r_sun)] = np.nan

# Keep clusters with valid coordinates
gc_data = merged_df.dropna(subset=["RA", "DEC"]).copy()

# Build coordinates from exactly these rows
gc_icrs = SkyCoord(
    ra=gc_data["RA"].to_numpy(dtype=float) * u.rad,
    dec=gc_data["DEC"].to_numpy(dtype=float) * u.rad,
    distance=gc_data["R_Sun"].to_numpy(dtype=float) * u.kpc,
    frame="icrs"
)

gc_sgr = gc_icrs.transform_to(sgr.frame)

# Calculate nearest angular separation for these same clusters
separations = gc_sgr[:, None].separation(sgr[None, :])
min_sep = separations.min(axis=1)

# Approximate projected distance in kpc
r_sun = gc_data["R_Sun"].to_numpy(dtype=float)
projected_dist = r_sun * min_sep.to_value(u.rad)

# Keep missing distances as NaN
projected_dist[~np.isfinite(r_sun)] = np.nan


# Plot
fig, ax = plt.subplots(
    figsize=(10, 5),
    subplot_kw={"projection": "aitoff"}
)

sc = ax.scatter(
    gc_sgr.Lambda.wrap_at(180 * u.deg).radian,
    gc_sgr.Beta.radian,
    c=projected_dist,
    cmap="viridis_r",
    s=55,
    edgecolors="black",
    linewidths=0.4,
    zorder=3
)

ax.plot(
    sgr.Lambda.wrap_at(180 * u.deg).radian,
    sgr.Beta.radian,
    ".",
    color="grey",
    markersize=2,
    label="Sagittarius stream"
)

cbar = plt.colorbar(sc, ax=ax, pad=0.08)
cbar.set_label("Approximate projected distance from stream (kpc)")

ax.set_title("Globular clusters: projected proximity to Sagittarius stream")
ax.grid(True)
ax.legend()
plt.tight_layout()
plt.show()

NameError: name 'projected_dist' is not defined